In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm

In [2]:
ENTREPOT_PATH = '/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/'
donnees = {}

def import_df(df_name, path_data, sep, index_col=None):
    donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) : 
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables = [
    'synthetise',
    'sdc',
    'domaine',
    'typologie_assol_can_realise',
    'typologie_can_rotation_synthetise',
    'entite_unique_par_sdc_nettoyage',
    'sdc_realise_performance',
    'synthetise_synthetise_performance'
]

# import des données du magasin
import_dfs(tables, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 8/8 [00:05<00:00,  1.44it/s]


In [ ]:
lst_alerte_col = ['alerte_ferti_n_tot', 'alerte_ift_cible_non_mil_chim_tot_hts',
       'alerte_ift_cible_non_mil_f', 'alerte_ift_cible_non_mil_h',
       'alerte_ift_cible_non_mil_i', 'alerte_ift_cible_non_mil_biocontrole',
       'alerte_co_irrigation_std_mil', 'alerte_msn_std_mil_avec_autoconso',
       'alerte_nombre_interventions_phyto', 'alerte_pb_std_mil_avec_autoconso',
       'alerte_rendement', 'alertes_charges', 'alerte_cm_std_mil',
       'alerte_co_semis_std_mil', 'alerte_tps_travail_total']

unique_entity = donnees['entite_unique_par_sdc_nettoyage']
sdc_real = donnees['sdc_realise_performance'][['sdc_id']+lst_alerte_col]
sdc_synth = donnees['synthetise_synthetise_performance'][['synthetise_id']+lst_alerte_col]
sdc = donnees['sdc'].rename(columns={'id':'sdc_id'})
synth = donnees['synthetise'].rename(columns={'id':'synthetise_id'})
typo_re = donnees['typologie_assol_can_realise']
typo_sy = donnees['typologie_can_rotation_synthetise']


sdc_real = sdc_real.merge(sdc, on='sdc_id', how='left').merge(typo_re, on='sdc_id', how='left')
sdc_synth = sdc_synth.merge(synth, on='synthetise_id', how='left').merge(sdc, on='sdc_id', how='left').merge(typo_sy, on='synthetise_id', how='left')

In [4]:
# Premierement on filtre selon la filière GCPE

sdc_real = sdc_real.loc[sdc_real['filiere'].isin(['POLYCULTURE_ELEVAGE','GRANDES_CULTURES'])]
sdc_synth = sdc_synth.loc[sdc_synth['filiere'].isin(['POLYCULTURE_ELEVAGE','GRANDES_CULTURES'])]

In [5]:
# Deuxiemement on filtre grace à l'outil d'entité unique par sdc
# il regarde dans un sdc s'il y a plusieurs synthétisé et ou zone de réalisé et en choisi un seul

lst_unqiue_entity_real = unique_entity.loc[unique_entity['entite_retenue']=='realise_retenu','sdc_id'].tolist()
sdc_real = sdc_real.loc[sdc_real['sdc_id'].isin(lst_unqiue_entity_real)]

lst_unqiue_entity_synth = unique_entity.loc[unique_entity['entite_retenue']!='realise_retenu','entite_retenue'].to_list()
sdc_synth = sdc_synth.loc[sdc_synth['synthetise_id'].isin(lst_unqiue_entity_synth)]

In [ ]:
# Troisiemement on va filtré l'année en cours pour le magasin car les données sont surement en cours de saisie ou en cours de consolidation par les IR
# En gros au 1° avril on accorde lajout de l'année n-1

# Définir les années limites (seuils strictes)
if datetime.now().month <= 3 :
    annees_max = datetime.now().year - 1
else : 
    annees_max = datetime.now().year
    
annees_trop_vieille = 2004 # des pz0 attendues jusqu'en 2005

sdc_synth = sdc_synth.loc[((pd.to_numeric(sdc_synth['campagne'], errors='coerce')) > annees_trop_vieille) & 
                          ((pd.to_numeric(sdc_synth['campagne'], errors='coerce')) < annees_max)]
sdc_real = sdc_real.loc[((pd.to_numeric(sdc_real['campagne'], errors='coerce')) > annees_trop_vieille) & 
                        ((pd.to_numeric(sdc_real['campagne'], errors='coerce')) < annees_max)]

In [ ]:
# Quatriemement on filtre par alertes
list_alerte_ok = ["Pas d'alerte", "Cette alerte n'existe pas dans cette filière", "Cette alerte n'existe pas encore dans cette filière"]

def filtrer_alertes(df, lst_alerte_col, list_alerte_ok, name_culture_col):
    # Masque pour les colonnes d'alertes sauf 1
    autres_colonnes = [col for col in lst_alerte_col if col != 'alerte_cm_std_mil']
    mask_autres = df[autres_colonnes].apply(
        lambda x: x.isin(list_alerte_ok) | x.isna()
    ).all(axis=1)

    # Mask pour les alertes de CM lorsque la culture est majoritairement de la prairie
    mask_cm = (
        df['alerte_cm_std_mil'].isin(list_alerte_ok) |
        df['alerte_cm_std_mil'].isna() |
        ((df['alerte_cm_std_mil'].str.contains('<', na=False)) & (df[name_culture_col] == "prairie temporaire >= 50 % assolement"))
        )

    mask_final = mask_autres & mask_cm
    return df[mask_final]


sdc_real = filtrer_alertes(sdc_real, lst_alerte_col, list_alerte_ok, 'typocan_assol_dvlp')
sdc_synth = filtrer_alertes(sdc_synth, lst_alerte_col, list_alerte_ok, 'typocan_rotation')

In [ ]:
# Au final on ne récupère qu'une liste en synthetise et une liste en réalisé des entités que l'on veut gardé
sdc_real['sdc_id']
sdc_synth['synthetise_id']

In [ ]:
########## PASSAGE AUX AGREGATIONS #################

In [ ]:
tables = [
    'synthetise',
    'sdc',
    'domaine',
    'typologie_assol_can_realise',
    'typologie_can_rotation_synthetise',
    'entite_unique_par_sdc_nettoyage',
    'sdc_realise_performance',
    'synthetise_synthetise_performance'
]

# import des données du magasin
import_dfs(tables, ENTREPOT_PATH, sep = ',', verbose=False)

In [ ]:
def list_to_scalar(serie):
    unique_values = list(serie.dropna().unique())
    if len(unique_values) == 0:
        return None
    elif len(unique_values) == 1:
        return unique_values[0]
    else:
        return unique_values

# Préparer un dataframe des sdc réalisé avec leur pz0 identifié
pz0ident_real = intervention_realise_agrege.merge(identification_pz0.rename(columns={'entite_id':'zone_id'}), on='zone_id', how='left')[['sdc_id','pz0']].groupby('sdc_id')['pz0'].apply(lambda x: list_to_scalar(x), include_groups=False).reset_index()
pz0ident_real = pz0ident_real.loc[pz0ident_real['pz0'].notna()]

if len(pz0ident_real.loc[pz0ident_real['pz0'].apply(lambda x: isinstance(x, list))] ) > 0 :
    raise ValueError("Il y a des sdc réalisé avec plusieurs identification différentes selon leur zones")

# On supprime les synthétisés ou réalisés n'ayant pas de pz0 identifié correctement après avoir merge les identifications
df = df.merge(identification_pz0.rename(columns={'entite_id':'synthetise_id'}), on='synthetise_id', how='left')
df = df.merge(pz0ident_real.rename(columns={'entite_id':'sdc_id'}), on='sdc_id', how='left')
df['pz0'] = df['pz0_x'].combine_first(df['pz0_y'])
df = df.drop(columns=['pz0_x','pz0_y'])

print(f"Il y a {len(df.loc[~df['pz0'].isin(['pz0','post'])])} synthétisé ou sdc filtrés n'ayant pas de pz0 identifié correctement.")

df = df.loc[df['pz0'].isin(['pz0','post'])]